In [ ]:
# 1) Check GPU
!nvidia-smi

Sat Jun  6 16:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 2) Install dependencies
!pip -q install -U uv

# Basic Python dependencies.
!uv pip install --system -U openai tqdm requests jsonschema psutil

# Install recent vLLM nightly for CUDA 13.0 / Blackwell.
# If this fails in your environment, use the auto backend line below instead.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check
import torch
import vllm
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 75ms
Checked 25 packages in 0.27ms
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 7.53s
Checked 190 packages in 1ms
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.22.1rc1.dev237+gfa27d4e9c


In [ ]:
# 3) Mount Google Drive and prepare paths
from google.colab import drive
from pathlib import Path
import shutil
import json
import os

drive.mount("/content/drive")

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
GDRIVE_INPUT_PATH = GDRIVE_PROJECT_DIR / "2wikimultihopqa_docs_chunks.json"

LOCAL_WORK_DIR = Path("/content/2wikimultihopqa_kg_test")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_INPUT_PATH = LOCAL_WORK_DIR / "2wikimultihopqa_docs_chunks.json"

KG_DIR = GDRIVE_PROJECT_DIR / "kg" / "2wikimultihopqa"
KG_DIR.mkdir(parents=True, exist_ok=True)

assert GDRIVE_INPUT_PATH.exists(), f"Input file not found: {GDRIVE_INPUT_PATH}"

# Copy to local disk for faster reading.
shutil.copy2(GDRIVE_INPUT_PATH, LOCAL_INPUT_PATH)

print("Input:", LOCAL_INPUT_PATH)
print("Output folder:", KG_DIR)

Mounted at /content/drive
Input: /content/2wikimultihopqa_kg_test/2wikimultihopqa_docs_chunks.json
Output folder: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa


In [ ]:
# 4) Start vLLM server - optimized throughput version
import subprocess
import time
import requests
from pathlib import Path
import os
import shlex
import psutil

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Total context length = prompt tokens + output tokens.
# 12288 is enough for ~1k-2k prompt + adaptive output up to 8192.
MAX_MODEL_LEN = 12288

# Keep memory margin on RTX PRO 6000 Blackwell 96GB.
GPU_MEMORY_UTILIZATION = 0.93

# Throughput settings.
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)
    except Exception:
        pass

# Stop old PID from previous run.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill any leftover vLLM server process.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode.
    "--language-model-only",

    # Qwen3 non-thinking mode at server level.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Higher concurrency for throughput.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Prefix caching should help because the prompt prefix is mostly shared.
    "--enable-prefix-caching",

    # Use vLLM default generation config, not HF generation_config.json.
    "--generation-config", "vllm",

    # Explicit dtype for Blackwell.
    "--dtype", "bfloat16",

    # Safe for HF repos that need model code.
    "--trust-remote-code",
]

# Do NOT add --enforce-eager.
# Keeping torch.compile / CUDA graph path enabled should improve throughput after startup.

server_env = os.environ.copy()

# Keep this fix: FlashInfer sampler crashes in this Blackwell environment.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# CUDA 13.0 / Blackwell runtime hints.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 12288 --gpu-memory-utilization 0.93 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 8 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 1849
Log: /content/vllm_server.log


In [ ]:
# 5) Wait for vLLM server with compact logging
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=2398) INFO 06-06 16:15:07 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=2398) INFO 06-06 16:15:07 [parallel_state.py:1568] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:48871 backend=nccl
(EngineCore pid=2398) INFO 06-06 16:15:07 [parallel_state.py:1903] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=2398) INFO 06-06 16:15:07 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=2398) INFO 06-06 16:15:08 [gpu_model_runner.py:5092] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=2398) INFO 06-06 16:15:08 [cuda.py:433] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=

In [ ]:
# 6) Load the first 100 chunks
import json
from pathlib import Path

with open(LOCAL_INPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

assert isinstance(chunks, list), "The input JSON must be a list of chunks."

TEST_LIMIT = 100
test_chunks = chunks[:TEST_LIMIT]

print("Total chunks:", len(chunks))
print("Test chunks:", len(test_chunks))
print(json.dumps(test_chunks[0], ensure_ascii=False, indent=2)[:2000])

Total chunks: 12685
Test chunks: 100
{
  "Chunk_id": "2wikimultihopqa_chunk_00000001",
  "Title": "Calloway County High School",
  "Paragraph_id": [
    1,
    2,
    3,
    4
  ],
  "Text": "Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\nOrganizations: Clubs/Organizations\nState champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball: 2004 Girls Golf: 2012 (Individual, Anna Hack)\nW. Earl Brown, actor: Pookie Jones, 1989 KHSAA Mr. Football winner*",
  "Token_count": 153
}


In [ ]:
# 7) Define schema and prompt
import json

KG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "entities": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "string",
                "minLength": 1
            }
        },
        "relations": {
            "type": "array",
            "maxItems": 35,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "head": {
                        "type": "string",
                        "minLength": 1
                    },
                    "relation": {
                        "type": "string",
                        "minLength": 1
                    },
                    "tail": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["head", "relation", "tail"]
            }
        },
        "facts": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "info": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["entity", "info"]
            }
        }
    },
    "required": ["entities", "relations", "facts"]
}

EXAMPLE_OUTPUT = {
    "entities": [
        "Marie Curie",
        "radioactivity",
        "Pierre Curie",
        "Curie Institute",
        "Paris",
        "1920"
    ],
    "relations": [
        {
            "head": "Marie Curie",
            "relation": "Marie Curie conducted pioneering research on radioactivity.",
            "tail": "radioactivity"
        },
        {
            "head": "Marie Curie",
            "relation": "Marie Curie was married to Pierre Curie.",
            "tail": "Pierre Curie"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in Paris.",
            "tail": "Paris"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in 1920.",
            "tail": "1920"
        }
    ],
    "facts": [
        {
            "entity": "Marie Curie",
            "info": "Marie Curie was a Polish and naturalized-French physicist and chemist who researched radioactivity."
        },
        {
            "entity": "Pierre Curie",
            "info": "Pierre Curie was married to Marie Curie."
        },
        {
            "entity": "Curie Institute",
            "info": "The Curie Institute in Paris was founded in 1920."
        }
    ]
}

def build_prompt(chunk):
    chunk_id = chunk["Chunk_id"]
    title = chunk["Title"]
    paragraph_ids = json.dumps(chunk["Paragraph_id"], ensure_ascii=False)
    chunk_text = chunk["Text"]

    return f"""You are an expert information extraction system designed to build a highly accurate retrieval knowledge graph.

Your task is to extract entities, relations between entities, and specific facts about entities from a given Wikipedia chunk.

Extract only facts explicitly supported by the chunk.
Do not use external knowledge.
Do not infer facts that are not clearly stated.
Prefer precision over recall.
Return JSON only.

Extract:

1. entities:
Important specific entities useful for multi-hop retrieval.
Include specific people, organizations, locations, works, events, awards, dates/years, and key concepts.
Do not extract generic adjectives or common nouns as standalone entities.
Extract at most 30 entities.

2. relations:
A relation is a connection between two extracted entities that is explicitly stated or directly supported by the chunk.
Each relation must have:
- head: one entity copied exactly from entities
- tail: one entity copied exactly from entities
- relation: a short, simple natural-language sentence explaining the connection between head and tail
Extract at most 35 relations.

3. facts:
A fact is a short, simple natural-language sentence describing what specific information this exact chunk provides about one extracted entity.
Each fact must have:
- entity: one entity copied exactly from entities
- info: a short sentence about that entity based only on this chunk
Extract at most 30 facts.

Rules:
- Every head and tail in relations must be copied exactly from the entities array.
- Every entity in facts must be copied exactly from the entities array.
- Do not create duplicate entities, duplicate relations, or duplicate facts.
- Resolve pronouns to actual entity names only when unambiguous.
- Generic words may appear inside relation and info sentences, but not as standalone entities.
- The output must be valid JSON strictly matching the required schema.

Example:

Input:
chunk_id:
ex_001

title:
Marie Curie

paragraph_ids:
[1]

text:
Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was married to Pierre Curie. The Curie Institute in Paris was founded in 1920.

Output:
{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}

Now extract from this chunk.

Input:
chunk_id:
{chunk_id}

title:
{title}

paragraph_ids:
{paragraph_ids}

text:
{chunk_text}

Output:
"""

In [ ]:
# 8) Define client and adaptive extraction helpers
import json
import time
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Adaptive output budget.
OUTPUT_TOKEN_STEPS = [4096, 8192]
MAX_OUTPUT_TOKENS = max(OUTPUT_TOKEN_STEPS)

# Qwen3.5 non-thinking general-task parameters.
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0

# For information extraction, 0.0 is safer than 1.5.
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.0

SAVE_RAW_RESPONSE_ON_ERROR = True

def make_messages(chunk):
    return [
        {
            "role": "system",
            "content": "You extract knowledge graph data from Wikipedia chunks. Return valid JSON only."
        },
        {
            "role": "user",
            "content": build_prompt(chunk)
        }
    ]

def usage_to_dict(usage):
    if usage is None:
        return None

    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

def call_llm_once(chunk, max_tokens):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=make_messages(chunk),
        max_tokens=max_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        presence_penalty=PRESENCE_PENALTY,
        seed=42,
        extra_body={
            "top_k": TOP_K,
            "min_p": MIN_P,
            "repetition_penalty": REPETITION_PENALTY,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": KG_SCHEMA
            }
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def extract_one_raw_adaptive(chunk, max_retries_per_budget=1):
    # Store exactly the model JSON fields after JSON parsing.
    start = time.perf_counter()
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None
    attempts = []

    for max_tokens in OUTPUT_TOKEN_STEPS:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                # If the model hit the token budget, retry with a larger budget.
                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                # Parse only to make final saved file valid JSON.
                model_output = json.loads(raw)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),

                    # Model output copied as-is after JSON parsing.
                    "entities": model_output.get("entities"),
                    "relations": model_output.get("relations"),
                    "facts": model_output.get("facts"),

                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

        # Continue to the next larger max_tokens budget.

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
    }

    if SAVE_RAW_RESPONSE_ON_ERROR:
        failed_item["raw_response"] = raw

    return failed_item

In [9]:
# 9) Run concurrent extraction for a resumable 5000-chunk batch
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time
import os
from datetime import datetime, timezone

# Batch config.
START_INDEX = 0          # Zero-based. Use 5000 for the next batch.
BATCH_SIZE = 5000
END_INDEX = min(START_INDEX + BATCH_SIZE, len(chunks))

batch_chunks = chunks[START_INDEX:END_INDEX]

# Output names include the exact chunk range.
BATCH_LABEL = (
    f"first{BATCH_SIZE}_chunks_{START_INDEX + 1:08d}_to_{END_INDEX:08d}"
    if START_INDEX == 0
    else f"chunks_{START_INDEX + 1:08d}_to_{END_INDEX:08d}"
)

OUTPUT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_{BATCH_LABEL}.json"
META_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_{BATCH_LABEL}_meta.json"
PARTIAL_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_{BATCH_LABEL}.partial.json"

MAX_WORKERS = 8
SAVE_EVERY = 25

overall_start = time.perf_counter()

results = [None] * len(batch_chunks)

def atomic_json_dump(obj, path):
    # Write safely, then replace.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def save_partial():
    # Save completed items only.
    partial_results = [x for x in results if x is not None]
    atomic_json_dump(partial_results, PARTIAL_PATH)

def load_existing_items():
    # Prefer final output, then partial checkpoint.
    for path in [OUTPUT_PATH, PARTIAL_PATH]:
        if path.exists():
            try:
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, list):
                    return data
            except Exception:
                pass

    return []

def make_failed_item(global_index, chunk, error):
    # Store failures without stopping the full batch.
    return {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": error,
        "latency_sec": None,
        "finish_reason": None,
        "max_tokens_used": None,
        "usage": None,
        "attempts": [],
        "input_index": global_index,
        "batch_local_index": global_index - START_INDEX,
        "batch_start_index": START_INDEX,
        "batch_end_index_exclusive": END_INDEX,
    }

# Resume from existing output or checkpoint.
existing_items = load_existing_items()

for item in existing_items:
    global_index = item.get("input_index")

    if global_index is None:
        continue

    local_index = global_index - START_INDEX

    if 0 <= local_index < len(results):
        results[local_index] = item

pending_jobs = [
    (START_INDEX + local_index, chunk)
    for local_index, chunk in enumerate(batch_chunks)
    if results[local_index] is None
]

def run_one(index_and_chunk):
    global_index, chunk = index_and_chunk

    item = extract_one_raw_adaptive(chunk, max_retries_per_budget=1)

    item["input_index"] = global_index
    item["batch_local_index"] = global_index - START_INDEX
    item["batch_start_index"] = START_INDEX
    item["batch_end_index_exclusive"] = END_INDEX

    return global_index, item

completed_new = 0
executor = None
cancelled = False

try:
    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

    futures = {
        executor.submit(run_one, job): job
        for job in pending_jobs
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f"Extracting KG {START_INDEX + 1}-{END_INDEX}"
    ):
        global_index, chunk = futures[future]
        local_index = global_index - START_INDEX

        try:
            _, item = future.result()
        except Exception as e:
            item = make_failed_item(global_index, chunk, repr(e))

        results[local_index] = item
        completed_new += 1

        if completed_new % SAVE_EVERY == 0:
            save_partial()

except KeyboardInterrupt:
    cancelled = True
    save_partial()

    if executor is not None:
        executor.shutdown(wait=False, cancel_futures=True)

    raise

finally:
    save_partial()

    if executor is not None and not cancelled:
        executor.shutdown(wait=True)

missing_indices = [
    START_INDEX + local_index
    for local_index, item in enumerate(results)
    if item is None
]

if missing_indices:
    raise RuntimeError(
        f"{len(missing_indices)} chunks are still missing. "
        f"Re-run this cell to resume from: {PARTIAL_PATH}"
    )

total_time_sec = time.perf_counter() - overall_start

num_errors = sum(1 for x in results if x["error"] is not None)
latencies = [
    x["latency_sec"]
    for x in results
    if x.get("latency_sec") is not None
]

chunks_per_sec = len(results) / total_time_sec if total_time_sec > 0 else None
chunks_per_min = chunks_per_sec * 60 if chunks_per_sec is not None else None

token_budget_counts = {}
finish_reason_counts = {}

for x in results:
    token_budget = x.get("max_tokens_used")
    token_budget_counts[str(token_budget)] = token_budget_counts.get(str(token_budget), 0) + 1

    finish_reason = x.get("finish_reason")
    finish_reason_counts[str(finish_reason)] = finish_reason_counts.get(str(finish_reason), 0) + 1

meta = {
    "dataset": "2wikimultihopqa",
    "input_path": str(GDRIVE_INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "partial_path": str(PARTIAL_PATH),
    "model": MODEL_NAME,

    "batch_start_index": START_INDEX,
    "batch_end_index_exclusive": END_INDEX,
    "batch_first_chunk_number": START_INDEX + 1,
    "batch_last_chunk_number": END_INDEX,
    "num_chunks": len(results),
    "num_errors": num_errors,

    "max_workers": MAX_WORKERS,
    "save_every": SAVE_EVERY,

    "server_max_num_seqs": MAX_NUM_SEQS,
    "server_max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "server_max_model_len": MAX_MODEL_LEN,
    "server_prefix_caching": True,
    "server_enforce_eager": False,

    "output_token_steps": OUTPUT_TOKEN_STEPS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "presence_penalty": PRESENCE_PENALTY,
    "repetition_penalty": REPETITION_PENALTY,

    "post_processing": "none",
    "token_budget_counts": token_budget_counts,
    "finish_reason_counts": finish_reason_counts,

    "total_time_sec": round(total_time_sec, 3),
    "avg_latency_sec": round(sum(latencies) / len(latencies), 3) if latencies else None,
    "chunks_per_min": round(chunks_per_min, 3) if chunks_per_min is not None else None,

    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

# Final saves.
atomic_json_dump(results, OUTPUT_PATH)
atomic_json_dump(meta, META_PATH)
save_partial()

print("Saved final JSON:", OUTPUT_PATH)
print("Saved checkpoint JSON:", PARTIAL_PATH)
print("Saved meta JSON:", META_PATH)

Extracting KG 1-5000:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved final JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_first5000_chunks_00000001_to_00005000.json
Saved checkpoint JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_first5000_chunks_00000001_to_00005000.partial.json
Saved meta JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_first5000_chunks_00000001_to_00005000_meta.json
